# Data integration

In [72]:
import pandas as pd
import numpy as np
import datetime as dt

In [37]:
path = "/home/fra/Documenti/BIC/Brazialian_Ecomm/data_processed/"

In [38]:
# customers
name = path + "olist_customers_dataset_cleaned.parquet"
customers = pd.read_parquet(name)

In [39]:
# geolocations
name = path + "olist_geolocation_dataset_cleaned.parquet"
geoloc = pd.read_parquet(name)

In [40]:
# items
name = path + "olist_order_items_dataset_cleaned.parquet"
items = pd.read_parquet(name)

In [41]:
# payments
name = path + "olist_order_payments_dataset_cleaned.parquet"
payments = pd.read_parquet(name)

In [42]:
# reviews
name = path + "olist_order_reviews_dataset_cleaned.parquet"
reviews = pd.read_parquet(name)

In [43]:
# orders
name = path + "olist_orders_dataset_cleaned.parquet"
orders = pd.read_parquet(name)

In [49]:
# products
name = path + "olist_products_dataset_cleaned.parquet"
products = pd.read_parquet(name)

In [20]:
# sellers
name = path + "olist_sellers_dataset_cleaned.parquet"
sellers = pd.read_parquet(name)

In [28]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [31]:
item_agg = items.groupby("order_id",as_index = False).agg(
    items_value = ("price","sum"),
    freight_value = ("freight_value","sum"),
    n_items = ("order_item_id","count"),
    n_sellers = ("seller_id","nunique")
)

In [32]:
item_agg.head()

,order_id,items_value,freight_value,n_items,n_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,1


# Joining the datasets

In [45]:
orders_full = (orders.merge(
    customers, on = "customer_id", how = "left", validate = "many_to_one").merge(
    item_agg, on = "order_id", how = "left", validate = "one_to_one").merge(
    reviews, on= "order_id", how=  "left", validate = "one_to_one")
)

In [52]:
orders_full.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,items_value,freight_value,n_items,n_sellers,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,03149,...,29.99,8.72,1.0,1.0,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,118.70,22.76,1.0,1.0,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,159.90,19.22,1.0,1.0,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18,2018-08-22 19:07:58
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,45.00,27.20,1.0,1.0,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03,2017-12-05 19:21:58
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,09195,...,19.90,8.72,1.0,1.0,e50934924e227544ba8246aeb3770dd4,5.0,NaN,NaN,2018-02-17,2018-02-18 13:02:51


In [54]:
orders_full.shape

(99441, 22)

In [50]:
items_full = (items.merge(
    products, on = "product_id", how = "left", validate = "many_to_one").merge(
    sellers, on = "seller_id", how = "left", validate = "many_to_one")
    )

In [55]:
items_full.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,03471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,87900,loanda,PR


In [51]:
items_full.shape

(112650, 18)

In [56]:
assert len(orders_full) == len(orders)
assert orders_full["order_id"].is_unique
orders_full.isna().sum()

order_id                             0
customer_id                          0
order_status                         0
order_purchase_timestamp             0
order_approved_at                  160
order_delivered_carrier_date      1783
order_delivered_customer_date     2965
order_estimated_delivery_date        0
customer_unique_id                   0
customer_zip_code_prefix             0
customer_city                        0
customer_state                       0
items_value                        775
freight_value                      775
n_items                            775
n_sellers                          775
review_id                          768
review_score                       768
review_comment_title             87890
review_comment_message           58658
review_creation_date               768
review_answer_timestamp            768
dtype: int64

In [74]:
orders_full.to_parquet(path + "orders_full.parquet", index=False)
items_full.to_parquet(path + "items_full.parquet", index=False)